# Import Library

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

from IPython.display import display


NumPy: 2.5.1
Pandas: 3.0.5
Device: cpu


# Load Dataset

In [2]:
# Dataset menggunakan delimiter ';' dan baris pertama adalah judul dataset
data = pd.read_csv(
    "gsar_a_2521295_sm1572.csv",
    sep=";",
    skiprows=1,
    encoding="utf-8-sig"
)

# Buang baris dengan SMILES kosong (jika ada) & pastikan label integer
data = data.dropna(subset=["SMILES", "label"]).reset_index(drop=True)
data["label"] = data["label"].astype(int)

print(data.shape)
data[["compound ID", "SMILES", "label"]].head()

(846, 8)


,compound ID,SMILES,label
0,A001,FC1(F)CN(c2c(C(F)(F)F)cnc3[nH]c(C4=CCNC4)cc23)C1,1
1,A002,CC(C)[C@@H](NC(=O)c1cc(-c2cnn3cc(-c4ccc(OCCN5C...,1
2,A003,O=C(NCC(F)(F)F)c1cc(-c2cnc3[nH]c(C4=CCCNC4)cc3...,1
3,A004,CN1CC=C(c2cc3cc(-c4csc(C(=O)NCC(F)(F)F)c4)cnc3...,1
4,A005,O=C(NCC(F)(F)F)c1cc(-c2cnc3[nH]c(C4=CCNCC4)cc3...,1


# Tokenisasi SMILES


In [3]:
X_smiles = data["SMILES"].values

# Bangun charset dari SELURUH SMILES di dataset + token start '!' dan end 'E'
charset = set("".join(list(X_smiles)) + "!E")
char_to_int = dict((c, i) for i, c in enumerate(charset))
int_to_char = dict((i, c) for i, c in enumerate(charset))

embed = max([len(smile) for smile in X_smiles]) + 5
vocab_size = len(charset)

print("Jumlah karakter unik (vocab_size):", vocab_size)
print("Panjang embedding (embed):", embed)
print(str(charset))

Jumlah karakter unik (vocab_size): 34
Panjang embedding (embed): 102
{'\\', 'O', 'F', '6', '2', 'l', '=', '@', 'P', 'c', 's', 'I', '1', '5', '#', '-', '(', 'E', 'C', 'o', 'S', 'r', ']', 'N', ')', '!', 'B', '4', 'H', '/', 'n', '3', '+', '['}


In [4]:
char_to_int

{'\\': 0,
 'O': 1,
 'F': 2,
 '6': 3,
 '2': 4,
 'l': 5,
 '=': 6,
 '@': 7,
 'P': 8,
 'c': 9,
 's': 10,
 'I': 11,
 '1': 12,
 '5': 13,
 '#': 14,
 '-': 15,
 '(': 16,
 'E': 17,
 'C': 18,
 'o': 19,
 'S': 20,
 'r': 21,
 ']': 22,
 'N': 23,
 ')': 24,
 '!': 25,
 'B': 26,
 '4': 27,
 'H': 28,
 '/': 29,
 'n': 30,
 '3': 31,
 '+': 32,
 '[': 33}

# Fungsi Vectorize

In [5]:
def vectorize(smiles, charset, char_to_int, embed):
    one_hot = np.zeros((smiles.shape[0], embed, len(charset)), dtype=np.int8)
    for i, smile in enumerate(smiles):
        # encode start char
        one_hot[i, 0, char_to_int["!"]] = 1
        # encode karakter SMILES
        for j, c in enumerate(smile):
            if c in char_to_int:
                one_hot[i, j + 1, char_to_int[c]] = 1
        # encode end char (padding sisa)
        one_hot[i, len(smile) + 1:, char_to_int["E"]] = 1
    return one_hot[:, 0:-1, :]

# Vectorize SELURUH dataset (tanpa split)
X_onehot = vectorize(X_smiles, charset, char_to_int, embed)

# Ubah one-hot menjadi integer sequence (siap untuk Embedding layer nantinya)
X_encoded = np.argmax(X_onehot, axis=2)

# Visual Hasil Encoding SMILES

In [6]:
print("vocab_size:", vocab_size, "| embed:", embed)
print("X_onehot shape :", X_onehot.shape)   # (jumlah_senyawa, panjang_sequence, jumlah_karakter_unik)
print("X_encoded shape:", X_encoded.shape)  # (jumlah_senyawa, panjang_sequence)

idx = 0  # ganti index sesuai molekul yang ingin dilihat

print("\nSMILES asli:", X_smiles[idx])
print()
print("One-hot encoding, 10 posisi pertama:")
print(X_onehot[idx][:10])

print("Integer-encoded, 20 posisi pertama:")
print(X_encoded[idx][:20])

decoded = "".join([int_to_char[i] for i in X_encoded[idx]])
print("Hasil decode balik:", decoded)

vocab_size: 34 | embed: 102
X_onehot shape : (846, 101, 34)
X_encoded shape: (846, 101)

SMILES asli: FC1(F)CN(c2c(C(F)(F)F)cnc3[nH]c(C4=CCNC4)cc23)C1

One-hot encoding, 10 posisi pertama:
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]
Integer-encoded, 20 posisi pertama:
[25  2 18 12 16  2 24 18 23 16  9  4  9 16 18 16  2 24 16  2]
Ha

# Penggabungan Hasil Encoding SMILES dengan Label

In [ ]:
y = data["label"].values

print("X_encoded shape:", X_encoded.shape)
print("y shape         :", y.shape)

# Validasi
assert X_encoded.shape[0] == y.shape[0], "Jumlah baris SMILES dan label tidak sinkron!"

X_encoded shape: (846, 101)
y shape         : (846,)


# Split Train/Test

In [ ]:
X = X_encoded          
y_arr = y.reshape(-1, 1) if y.ndim == 1 else y

X_train, X_test, Y_train, Y_test = train_test_split(
    X, y_arr,
    test_size=0.3,
    random_state=42,
    stratify=y_arr
)

print(X_train.shape, X_test.shape)

(592, 101) (254, 101)


# Dataset & Dataset Loader

In [9]:
class SmilesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = SmilesDataset(X_train, Y_train)
test_dataset  = SmilesDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Pembentukan model Baseline

In [10]:
class LSTMBaseline(nn.Module):
    def __init__(self, vocab_size, embedding_dim=50, units_list=[64]):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm_layers = nn.ModuleList()
        input_size = embedding_dim
        for units in units_list:
            self.lstm_layers.append(nn.LSTM(input_size=input_size, hidden_size=units, batch_first=True))
            input_size = units
        self.output_layer = nn.Linear(units_list[-1], 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.embedding(x)
        for lstm in self.lstm_layers:
            out, _ = lstm(out)
        out = out[:, -1, :]
        out = self.output_layer(out)
        return self.sigmoid(out)


def train_model(model, train_loader, test_loader, epochs=50, lr=0.001, device=device):
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
            train_correct += ((preds > 0.5).float() == yb).sum().item()
            train_total += xb.size(0)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
                val_correct += ((preds > 0.5).float() == yb).sum().item()
                val_total += xb.size(0)

        history["loss"].append(train_loss / train_total)
        history["accuracy"].append(train_correct / train_total)
        history["val_loss"].append(val_loss / val_total)
        history["val_accuracy"].append(val_correct / val_total)

        print(f"Epoch {epoch+1}/{epochs} - loss: {history['loss'][-1]:.4f} - acc: {history['accuracy'][-1]:.4f} "
              f"- val_loss: {history['val_loss'][-1]:.4f} - val_acc: {history['val_accuracy'][-1]:.4f}")

    return history


def get_predictions(model, data_loader, device=device, threshold=0.5):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(device)
            probs = model(xb).cpu().numpy().flatten()
            preds = (probs >= threshold).astype(int)
            y_true.extend(yb.numpy().flatten().astype(int))
            y_pred.extend(preds)
    return np.array(y_true), np.array(y_pred)


def compute_confusion_matrix(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP, TN, FP, FN


def compute_metrics(TP, TN, FP, FN, eps=1e-10):
    accuracy  = (TP + TN) / (TP + TN + FP + FN + eps)
    recall    = TP / (TP + FN + eps)
    precision = TP / (TP + FP + eps)
    f1_score  = 2 * (precision * recall) / (precision + recall + eps)
    return {"accuracy": accuracy, "recall": recall, "precision": precision, "f1_score": f1_score}


def print_confusion_matrix(TP, TN, FP, FN, model_name=""):
    cm_df = pd.DataFrame(
        [[TP, FN], [FP, TN]],
        index=["Aktual Aktif (1)", "Aktual Inaktif (0)"],
        columns=["Prediksi Aktif (1)", "Prediksi Inaktif (0)"]
    )
    print(f"\nConfusion Matrix - {model_name}")
    print(cm_df)
    return cm_df

# Baseline 1

In [11]:
print("="*60)
print("BASELINE 1 | LSTM layers: [64]")
print("="*60)

model_b1 = LSTMBaseline(vocab_size=vocab_size, embedding_dim=50, units_list=[64])
print(model_b1)

history_b1 = train_model(model_b1, train_loader, test_loader, epochs=50, lr=0.001)

BASELINE 1 | LSTM layers: [64]
LSTMBaseline(
  (embedding): Embedding(34, 50)
  (lstm_layers): ModuleList(
    (0): LSTM(50, 64, batch_first=True)
  )
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Epoch 1/50 - loss: 0.6952 - acc: 0.4561 - val_loss: 0.6934 - val_acc: 0.4961
Epoch 2/50 - loss: 0.6939 - acc: 0.4932 - val_loss: 0.6927 - val_acc: 0.5118
Epoch 3/50 - loss: 0.6963 - acc: 0.4882 - val_loss: 0.6936 - val_acc: 0.4961
Epoch 4/50 - loss: 0.6939 - acc: 0.5000 - val_loss: 0.6886 - val_acc: 0.5354
Epoch 5/50 - loss: 0.6621 - acc: 0.6841 - val_loss: 0.6081 - val_acc: 0.7362
Epoch 6/50 - loss: 0.5328 - acc: 0.7652 - val_loss: 0.4817 - val_acc: 0.7953
Epoch 7/50 - loss: 0.4790 - acc: 0.7720 - val_loss: 0.4345 - val_acc: 0.8189
Epoch 8/50 - loss: 0.4292 - acc: 0.8159 - val_loss: 0.3905 - val_acc: 0.8346
Epoch 9/50 - loss: 0.4051 - acc: 0.8361 - val_loss: 0.3755 - val_acc: 0.8543
Epoch 10/50 - loss: 0.4071 - acc: 0.8429 - val_loss: 0.3614 - v

In [12]:
y_true_b1, y_pred_b1 = get_predictions(model_b1, test_loader)
TP1, TN1, FP1, FN1 = compute_confusion_matrix(y_true_b1, y_pred_b1)
metrics_b1 = compute_metrics(TP1, TN1, FP1, FN1)

print_confusion_matrix(TP1, TN1, FP1, FN1, model_name="Baseline 1")
print(f"\nAccuracy  : {metrics_b1['accuracy']:.4f}")
print(f"Recall    : {metrics_b1['recall']:.4f}")
print(f"Precision : {metrics_b1['precision']:.4f}")
print(f"F1-Score  : {metrics_b1['f1_score']:.4f}")


Confusion Matrix - Baseline 1
                    Prediksi Aktif (1)  Prediksi Inaktif (0)
Aktual Aktif (1)                   101                    25
Aktual Inaktif (0)                   6                   122

Accuracy  : 0.8780
Recall    : 0.8016
Precision : 0.9439
F1-Score  : 0.8670


# BAseline 2

In [13]:
print("="*60)
print("BASELINE 2 | LSTM layers: [64, 128]")
print("="*60)

model_b2 = LSTMBaseline(vocab_size=vocab_size, embedding_dim=50, units_list=[64, 128])
print(model_b2)

history_b2 = train_model(model_b2, train_loader, test_loader, epochs=50, lr=0.001)

BASELINE 2 | LSTM layers: [64, 128]
LSTMBaseline(
  (embedding): Embedding(34, 50)
  (lstm_layers): ModuleList(
    (0): LSTM(50, 64, batch_first=True)
    (1): LSTM(64, 128, batch_first=True)
  )
  (output_layer): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Epoch 1/50 - loss: 0.6939 - acc: 0.5051 - val_loss: 0.6926 - val_acc: 0.5157
Epoch 2/50 - loss: 0.6897 - acc: 0.5355 - val_loss: 0.6363 - val_acc: 0.7520
Epoch 3/50 - loss: 0.5930 - acc: 0.7230 - val_loss: 0.4720 - val_acc: 0.7874
Epoch 4/50 - loss: 0.5809 - acc: 0.6875 - val_loss: 0.5300 - val_acc: 0.7874
Epoch 5/50 - loss: 0.5058 - acc: 0.7652 - val_loss: 0.4734 - val_acc: 0.7874
Epoch 6/50 - loss: 0.4794 - acc: 0.7821 - val_loss: 0.4505 - val_acc: 0.7913
Epoch 7/50 - loss: 0.4445 - acc: 0.8024 - val_loss: 0.4256 - val_acc: 0.8110
Epoch 8/50 - loss: 0.4290 - acc: 0.8057 - val_loss: 0.4321 - val_acc: 0.8031
Epoch 9/50 - loss: 0.4366 - acc: 0.8176 - val_loss: 0.4480 - val_acc: 0.7992
Epoch 10/50 - lo

In [14]:
y_true_b2, y_pred_b2 = get_predictions(model_b2, test_loader)
TP2, TN2, FP2, FN2 = compute_confusion_matrix(y_true_b2, y_pred_b2)
metrics_b2 = compute_metrics(TP2, TN2, FP2, FN2)

print_confusion_matrix(TP2, TN2, FP2, FN2, model_name="Baseline 2")
print(f"\nAccuracy  : {metrics_b2['accuracy']:.4f}")
print(f"Recall    : {metrics_b2['recall']:.4f}")
print(f"Precision : {metrics_b2['precision']:.4f}")
print(f"F1-Score  : {metrics_b2['f1_score']:.4f}")


Confusion Matrix - Baseline 2
                    Prediksi Aktif (1)  Prediksi Inaktif (0)
Aktual Aktif (1)                   108                    18
Aktual Inaktif (0)                  13                   115

Accuracy  : 0.8780
Recall    : 0.8571
Precision : 0.8926
F1-Score  : 0.8745


# Baseline 3

In [15]:
print("="*60)
print("BASELINE 3 | LSTM layers: [64, 128, 256]")
print("="*60)

model_b3 = LSTMBaseline(vocab_size=vocab_size, embedding_dim=50, units_list=[64, 128, 256])
print(model_b3)

history_b3 = train_model(model_b3, train_loader, test_loader, epochs=50, lr=0.001)

BASELINE 3 | LSTM layers: [64, 128, 256]
LSTMBaseline(
  (embedding): Embedding(34, 50)
  (lstm_layers): ModuleList(
    (0): LSTM(50, 64, batch_first=True)
    (1): LSTM(64, 128, batch_first=True)
    (2): LSTM(128, 256, batch_first=True)
  )
  (output_layer): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Epoch 1/50 - loss: 0.7010 - acc: 0.4764 - val_loss: 0.6929 - val_acc: 0.5079
Epoch 2/50 - loss: 0.6933 - acc: 0.5169 - val_loss: 0.6924 - val_acc: 0.6654
Epoch 3/50 - loss: 0.6769 - acc: 0.5743 - val_loss: 0.5895 - val_acc: 0.7047
Epoch 4/50 - loss: 0.5654 - acc: 0.7128 - val_loss: 0.5008 - val_acc: 0.7598
Epoch 5/50 - loss: 0.4994 - acc: 0.7770 - val_loss: 0.4543 - val_acc: 0.7992
Epoch 6/50 - loss: 0.4544 - acc: 0.8007 - val_loss: 0.4490 - val_acc: 0.7953
Epoch 7/50 - loss: 0.4477 - acc: 0.7922 - val_loss: 0.5197 - val_acc: 0.6890
Epoch 8/50 - loss: 0.4704 - acc: 0.7652 - val_loss: 0.4171 - val_acc: 0.8189
Epoch 9/50 - loss: 0.4280 - acc: 0.7990 - val_

In [16]:
y_true_b3, y_pred_b3 = get_predictions(model_b3, test_loader)
TP3, TN3, FP3, FN3 = compute_confusion_matrix(y_true_b3, y_pred_b3)
metrics_b3 = compute_metrics(TP3, TN3, FP3, FN3)

print_confusion_matrix(TP3, TN3, FP3, FN3, model_name="Baseline 3")
print(f"\nAccuracy  : {metrics_b3['accuracy']:.4f}")
print(f"Recall    : {metrics_b3['recall']:.4f}")
print(f"Precision : {metrics_b3['precision']:.4f}")
print(f"F1-Score  : {metrics_b3['f1_score']:.4f}")


Confusion Matrix - Baseline 3
                    Prediksi Aktif (1)  Prediksi Inaktif (0)
Aktual Aktif (1)                   101                    25
Aktual Inaktif (0)                   5                   123

Accuracy  : 0.8819
Recall    : 0.8016
Precision : 0.9528
F1-Score  : 0.8707


# Penggabungan Hasil ke 1 Dataframe

In [22]:
evaluation_results = {
    "Baseline 1": {"TP": TP1, "TN": TN1, "FP": FP1, "FN": FN1, **metrics_b1},
    "Baseline 2": {"TP": TP2, "TN": TN2, "FP": FP2, "FN": FN2, **metrics_b2},
    "Baseline 3": {"TP": TP3, "TN": TN3, "FP": FP3, "FN": FN3, **metrics_b3},
}

summary_df = pd.DataFrame(evaluation_results).T
summary_df = summary_df[["TP", "TN", "FP", "FN", "accuracy", "recall", "precision", "f1_score"]]
display(summary_df)

,TP,TN,FP,FN,accuracy,recall,precision,f1_score
Baseline 1,101.0,122.0,6.0,25.0,0.877953,0.801587,0.943925,0.866953
Baseline 2,108.0,115.0,13.0,18.0,0.877953,0.857143,0.892562,0.874494
Baseline 3,101.0,123.0,5.0,25.0,0.881890,0.801587,0.952830,0.870690


# SUMMary

In [23]:

histories = {"Baseline 1": history_b1, "Baseline 2": history_b2, "Baseline 3": history_b3}

training_summary_df = pd.DataFrame({
    name: {
        "train_acc": hist["accuracy"][-1],
        "val_acc": hist["val_accuracy"][-1],
        "train_loss": hist["loss"][-1],
        "val_loss": hist["val_loss"][-1],
    }
    for name, hist in histories.items()
}).T
display(training_summary_df)


,train_acc,val_acc,train_loss,val_loss
Baseline 1,0.930743,0.877953,0.207206,0.327206
Baseline 2,0.912162,0.877953,0.261041,0.329819
Baseline 3,0.917230,0.881890,0.236485,0.377167
